In [2]:
import requests
import time
import pandas as pd
import os

In [2]:
def filter_european(score):
    is_european = False
    temp = score.get("samples_training", [])
    if len(temp) > 0:
        ancestry = temp[0].get("ancestry_broad", [])
        if ancestry == "European":
            is_european = True
        else:
            pass
    if not is_european:
        temp = score.get("samples_variants", [])
        if len(temp) > 0:
            ancestry = temp[0].get("ancestry_broad", [])
            if ancestry == "European":
                is_european = True
            else:
                pass
    return is_european

def get_url(score):
    temp = score.get("ftp_harmonized_scoring_files", [])
    if len(temp) > 0:
        url = temp.get("GRCh38", []).get("positions", [])
        return url
    return False

def get_associdated_pgs_ids(score):
    temp = score.get("associated_pgs_ids", [])
    if len(temp) > 0:
        return temp
    return False

In [ ]:
# Read contained icd and description
icd2dsp = pd.read_csv(os.path.join(os.getcwd(), "trait_list_260217.csv"), index_col=None)

# generate icd root col
icd2dsp["icd_root"] = icd2dsp["icd"].astype(str).str[:3]
cols = list(icd2dsp.columns)
cols.remove("icd_root")
cols.insert(3, "icd_root")
icd2dsp = icd2dsp[cols]

base_url = "https://www.pgscatalog.org/rest/trait/search"

icd2dsp["pgs_ids"] = None
icd2dsp["pgs_api_num"] = None
icd2dsp["pgs_urls"] = None
for idx, row in icd2dsp.iterrows():
    time.sleep(5)
    icd = row["icd"] 
    description = row["ontology"]

    # get all pgs ids for this trait
    params = {"term": description}
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"Failed for {icd}: {description}")
        continue
    
    data = response.json()
    url = response.url  

    pgs_ids = []
    while True:
        for s in data["results"]:
            temp = get_associdated_pgs_ids(s)
            if temp:
                pgs_ids.extend(temp)

        next_url = data.get("next")
        if not next_url:
            break
        
        
        r = requests.get(next_url)
        if r.status_code != 200:
            print(f"Pagination failed for {icd}")
            break
        data = r.json()

    pgs_ids = list(set(pgs_ids))
    icd2dsp.at[idx, "pgs_ids"] = pgs_ids
    icd2dsp.at[idx, "pgs_api_num"] = int(len(pgs_ids))
    
    pgs_urls = []
    for pgs_id in pgs_ids:
        temp = "https://www.pgscatalog.org/rest/score/" + pgs_id
        r = requests.get(temp)
        data = r.json()

        # filter european pgs and get corresponding url
        # if filter_european(data):
        #     euro_url = get_url(data)
        #     pgs_urls.extend([euro_url])
        #     time.sleep(0.2)
        
        # get ll pgs`s corresponding url
        temp_url = get_url(data)
        pgs_urls.extend([temp_url])
        time.sleep(0.2)
    
    icd2dsp.at[idx, "pgs_urls"] = pgs_urls
    print(f"Finish scrolling {icd} with num of pgs:", len(pgs_ids), ", num of pgs urls:", len(pgs_urls))
    time.sleep(0.2)

icd2dsp.to_csv(os.path.join(os.getcwd(),"pgs_id_list_260217.csv"), index=False)

Finish scrolling I714 with num of pgs: 6 , num of pgs urls: 6
Finish scrolling R9431 with num of pgs: 1 , num of pgs urls: 1
Finish scrolling M0579 with num of pgs: 1 , num of pgs urls: 1
Finish scrolling M069 with num of pgs: 1 , num of pgs urls: 1
Finish scrolling N179 with num of pgs: 2 , num of pgs urls: 2
Finish scrolling C9100 with num of pgs: 9 , num of pgs urls: 9
Finish scrolling I219 with num of pgs: 2 , num of pgs urls: 2
Finish scrolling H905 with num of pgs: 2 , num of pgs urls: 2
Finish scrolling H353131 with num of pgs: 6 , num of pgs urls: 6
Finish scrolling F1020 with num of pgs: 4 , num of pgs urls: 4
Finish scrolling F1099 with num of pgs: 2 , num of pgs urls: 2
Finish scrolling J309 with num of pgs: 2 , num of pgs urls: 2
Finish scrolling G309 with num of pgs: 53 , num of pgs urls: 53
Finish scrolling D649 with num of pgs: 1 , num of pgs urls: 1
Finish scrolling I2510 with num of pgs: 19 , num of pgs urls: 19
Finish scrolling M459 with num of pgs: 9 , num of pgs url

JSONDecodeError: Expecting value: line 2 column 1 (char 1)

In [ ]:
import requests
import time

url = "https://www.pgscatalog.org/rest/score/all"

euro_urls = []

while url:
    r = requests.get(url)
    data = r.json()
    
    euro_data = [s for s in data["results"] if filter_european(s)]
    euro_url = [get_url(s) for s in euro_data if get_url(s)]
    euro_urls.extend(euro_url)
    
    url = data["next"]
    time.sleep(0.2)   

print("Total European GRCh38 URLs:", len(euro_urls))

Total European GRCh38 URLs: 2334


In [44]:
import pandas as pd
import os

df = pd.DataFrame({"url": euro_urls})
df.to_csv(os.path.join(os.getcwd(),"european_grch38_urls.csv"), index=False)

In [40]:
print(euro_urls)

[['https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000001/ScoringFiles/Harmonized/PGS000001_hmPOS_GRCh38.txt.gz', 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000002/ScoringFiles/Harmonized/PGS000002_hmPOS_GRCh38.txt.gz', 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000003/ScoringFiles/Harmonized/PGS000003_hmPOS_GRCh38.txt.gz', 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000004/ScoringFiles/Harmonized/PGS000004_hmPOS_GRCh38.txt.gz', 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000005/ScoringFiles/Harmonized/PGS000005_hmPOS_GRCh38.txt.gz', 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000006/ScoringFiles/Harmonized/PGS000006_hmPOS_GRCh38.txt.gz', 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000007/ScoringFiles/Harmonized/PGS000007_hmPOS_GRCh38.txt.gz', 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000008/ScoringFiles/Harmonized/PGS000008_hmPOS_GRCh38.txt.gz', 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/

In [2]:
temp = "https://www.pgscatalog.org/rest/score/PGS003852"
r = requests.get(temp)
data = r.json()

In [3]:
print(data)

{'id': 'PGS003852', 'name': 'CRC_PRS_EUR_EAS', 'ftp_scoring_file': 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS003852/ScoringFiles/PGS003852.txt.gz', 'ftp_harmonized_scoring_files': {'GRCh37': {'positions': 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS003852/ScoringFiles/Harmonized/PGS003852_hmPOS_GRCh37.txt.gz'}, 'GRCh38': {'positions': 'https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS003852/ScoringFiles/Harmonized/PGS003852_hmPOS_GRCh38.txt.gz'}}, 'publication': {'id': 'PGP000492', 'title': 'Combining Asian and European genome-wide association studies of colorectal cancer improves risk prediction across racial and ethnic populations.', 'doi': '10.1038/s41467-023-41819-0', 'PMID': 37783704, 'journal': 'Nat Commun', 'firstauthor': 'Thomas M', 'date_publication': '2023-10-02'}, 'matches_publication': True, 'samples_variants': [{'sample_number': 69175, 'sample_cases': None, 'sample_controls': None, 'sample_percent_male': None, 'sample_age': None, 'phenotypin